# Studio 2 — Pelatihan di Colab / Kaggle (runner)

Notebook ini **tipis**: semua logika ada di repo (`train.py`, `make_split.py`, `evaluate.py`), notebook hanya menyiapkan GPU + dataset lalu menjalankan skrip yang sama dengan yang dipakai asisten.

Batas dari halaman studio: `model.pth` **≤ 500 MB**, evaluasi seluruh validation set **≤ 15 menit**; bobot > 100 MB harus lewat GitHub Release / LFS. Yang dinilai asisten = **commit terakhir di branch utama sebelum tenggat**.

Alur: mount Drive -> clone repo -> temukan `train/` -> **cek split beku** -> latih -> uji `evaluate.py` -> unduh `submission/`.


In [ ]:
REPO_URL = "https://github.com/stcahya/DL02-klasifikasi-sampah.git"
BRANCH   = "main"

# Folder Drive hasil ekstrak dataset dari folder kelas (harus berisi train/).
# Sesuaikan path ini dengan Drive Anda.
DRIVE_ROOT = "/content/drive/MyDrive/Studio2-DeepLearning"

EPOCHS, BATCH_SIZE, WORKERS = 20, 64, 4

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content
![ -d repo/.git ] || git clone -b {BRANCH} {REPO_URL} repo
!cd repo && git pull --ff-only
%cd /content/repo
!pip install -q -r submission/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "TANPA GPU - latih di Kaggle atau lokal, atau set Runtime > GPU"
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"


### Kaggle

Sama saja, dua beda: **jangan** mount Drive — dataset di-attach sebagai input dan muncul di `/kaggle/input/<slug>/`, jadi set `DRIVE_ROOT` ke path itu. Kalau paket deps tidak bisa dipasang (internet off), aktifkan *Internet* di pengaturan notebook.

In [ ]:
import pathlib

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
# Jumlah gambar snapshot dataset yang dipakai membekukan splits/val.txt (make_split --check harus [OK]).
EXPECT = {"botol_plastik": 523, "gelas_plastik": 521, "kertas_kardus": 365,
          "saset_kemasan": 673, "styrofoam": 671}

root = pathlib.Path(DRIVE_ROOT)
if not root.exists():
    raise SystemExit(f"[GAGAL] {root} tidak ada - isi DRIVE_ROOT dengan folder dataset Anda.")

found = [d for d in root.rglob("train")
         if d.is_dir() and all((d / c).is_dir() for c in CLASSES)]
if not found:
    print("isi", root, ":", [str(p.relative_to(root)) for p in list(root.iterdir())[:20]])
    raise SystemExit("[GAGAL] tidak ketemu folder train/ berisi 5 kelas di bawah DRIVE_ROOT.")

src = max(found, key=lambda d: sum(len(list((d / c).glob("*.jpg"))) for c in CLASSES))
counts = {c: len(list((src / c).glob("*.jpg"))) for c in CLASSES}
print("dataset:", src, "| total", sum(counts.values()), "gambar")
beda = [c for c in CLASSES if counts[c] != EXPECT[c]]
for c in CLASSES:
    print(f"  {c:16s} {counts[c]:5d}  {'' if counts[c] == EXPECT[c] else '<-- beda dari snapshot split (%d)' % EXPECT[c]}")
if beda:
    print("\n[PERINGATAN] snapshot dataset ini beda dari yang dipakai membekukan splits/val.txt.")
    print("Kalau jumlahnya beda, make_split.py --check akan gagal - itu memang benar. Angka hasil")
    print("tidak akan sebanding dengan laptop lain. Pakai dulu snapshot yang sama (2753 gambar).")
else:
    print("\n[OK] snapshot dataset sama dengan yang membekukan splits/val.txt.")

link = pathlib.Path("train")
if link.is_symlink():
    link.unlink()
if not link.exists():
    link.symlink_to(src, target_is_directory=True)
print("train/ ->", link.resolve())

In [ ]:
!python make_split.py --check

`make_split.py --check` wajib `[OK]` (413 val / 2340 train, sha256 kanonik `b56ef897...`). Kalau `[GAGAL]`/`[BEDA]`, jangan lanjut melatih: berarti datasetnya beda dan hasilnya tidak bisa dibandingkan dengan laptop lain.

In [ ]:
!python train.py --epochs {EPOCHS} --bs {BATCH_SIZE} --workers {WORKERS}

`train.py` menulis `submission/model.pth` (checkpoint terbaik) + `my_val/` + `runs/last.json`, dan memilih checkpoint dengan macro-F1 memakai metrik + urutan kelas dari `evaluate.py`.

In [ ]:
!python evaluate.py --submission ./submission --data ./my_val --out runs/eval_colab.json

In [ ]:
import hashlib, pathlib, shutil

mp = pathlib.Path("submission/model.pth")
print(f"model.pth: {mp.stat().st_size / 2**20:.1f} MB (batas teknis 500 MB)")
print("sha256:", hashlib.sha256(mp.read_bytes()).hexdigest())

shutil.make_archive("/content/submission_model", "zip", "submission")
from google.colab import files
files.download("/content/submission_model.zip")

### Setelah selesai

Di laptop, dari folder repo:

```bash
git pull --ff-only
# ekstrak submission_model.zip menimpa submission/
git add submission/model.py submission/model.pth
git commit -m "submission: ResNet-18 <hasil>"
git push
python evaluate.py --submission ./submission --data ./my_val   # verifikasi ulang sebelum tenggat
```

Jangan mengubah `submission/model.py` setelah `model.pth` dibuat: `evaluate.py` memuat dengan `strict=True`, jadi ketidakcocokan arsitektur = submission gagal dievaluasi (skor 0 di putaran itu).
